# IN11: Benchmark Design, Regression Testing and LLM-as-Judge

## Objectives

By the end of this notebook you will be able to:

- Design a production-grade golden dataset with principled category coverage
- Implement an LLM-as-judge pipeline using a structured 0-3 rubric
- Run a baseline evaluation and a comparison evaluation across two model versions
- Detect performance regression and make a go/no-go deployment decision
- Understand score delta analysis and drift threshold setting

**Deliverable:** `in11_benchmark_report.json`

In [1]:
# Main problem being solved

# Suppose Walmart is currently using one model and wants to replace it with a newer or cheaper model.

# The company cannot deploy the new model only because it is faster or less expensive. It must first verify:
# - Are the new answers still accurate?
# - Did any category become weaker?
# - Is the response time better or worse?
# - Did token usage and cost increase?
# - Should the new model be approved, reviewed further, or blocked?

In [2]:
# Notebook workflow:

# 1. LLM-as-Judge: A separate LLM acts like an evaluator.
# 2. Benchmark dataset
# 3. Baseline evaluation
# 4. Comparison evaluation
# 5. Regression testing: Delta = Comparison score − Baseline score
# 6. Go/No-Go decision
# 7. Comparative judging

In [3]:
# Why should the same benchmark questions and judge model be used for both the baseline and comparison runs?

# A. To reduce API cost
# B. To ensure a fair comparison
# C. To make the answers longer
# D. To increase model temperature

In [4]:
import os, json, time, statistics
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)
print('Client ready.')

Client ready.


## Section 1: LLM-as-Judge

### What is LLM-as-Judge?

LLM-as-judge is an evaluation technique where a separate, trusted LLM is used to score
the output of the system under test. The judge receives the query, the context,
the expected answer, and the actual answer -- then returns a structured quality score.

**Why use LLM-as-judge over human evaluation?**

| Dimension | Human Evaluation | LLM-as-Judge |
|---|---|---|
| Speed | Days to weeks | Seconds |
| Cost | Very high | Low (API cost only) |
| Consistency | Variable (inter-rater disagreement) | Deterministic at temperature=0 |
| Scale | Limited | Unlimited |
| Nuance | High | High (with good rubric) |

**The 0-3 scoring rubric (Walmart Retail Assistant):**

| Score | Label | Criteria |
|---|---|---|
| 0 | Wrong / Harmful | Factually incorrect, or contains harmful content, or completely off-topic |
| 1 | Partially Correct | Correct intent but missing key facts, incomplete, or misleading |
| 2 | Correct but Incomplete | Factually correct but missing detail that a customer needs |
| 3 | Complete and Correct | Factually accurate, complete, appropriately concise |

**Normalised score = raw_score / 3**, giving a 0.0-1.0 scale compatible with other metrics.

**Production threshold:** Normalised score >= 0.67 (equivalent to raw >= 2 out of 3).

**What to remember:**
- Always set judge temperature to 0. Any randomness in the judge score invalidates comparison.
- Use the same judge model for baseline and comparison runs. Changing the judge changes the ruler.
- Be aware of positional bias: some LLMs tend to prefer the first answer shown.
  Mitigate by randomising the order of answers in comparative evaluations.
- Be aware of verbosity bias: some LLMs score longer answers higher regardless of quality.
  Counter this by including answer length in the rubric explicitly.
- The rubric must be written for the specific domain. A generic rubric produces
  unreliable scores for specialised tasks like retail pricing queries.

In [5]:
JUDGE_MODEL   = 'gpt-4-turbo'
JUDGE_TEMP    = 0

JUDGE_RUBRIC = (
    'Score the assistant response using this rubric:\n'
    '  0 = Wrong or harmful: factually incorrect, harmful, completely off-topic, or refuses a valid question\n'
    '  1 = Partially correct: right intent but missing key facts, incomplete, or potentially misleading\n'
    '  2 = Correct but incomplete: all facts are right but missing detail the customer needs\n'
    '  3 = Complete and correct: accurate, complete, and appropriately concise for a retail assistant\n'
    'Return JSON: {"score": 0-3, "normalised": 0.0-1.0, "label": "...", "reason": "one sentence"}'
)

def llm_judge(query: str, context: str, expected: str, actual: str) -> dict:
    """Score an actual answer against expected using LLM-as-judge."""
    prompt = (
        f'Query: {query}\n\n'
        f'Context (ground truth): {context}\n\n'
        f'Expected answer: {expected}\n\n'
        f'Actual answer: {actual}\n\n'
        + JUDGE_RUBRIC
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a strict retail AI quality evaluator. Follow the rubric exactly.'},
            {'role': 'user',   'content': prompt},
        ],
        temperature=JUDGE_TEMP,
        response_format={'type': 'json_object'},
    )
    result = json.loads(resp.choices[0].message.content)
    return {
        'score':      result.get('score', 0),
        'normalised': round(result.get('score', 0) / 3, 3), # normalise to 0.0-1.0
        'label':      result.get('label', ''), # e.g., "Correct but incomplete"
        'reason':     result.get('reason', ''), # one sentence explanation
    }

# Demo: judge a correct answer vs a hallucinated answer
demo = {
    'query':    'What is the price of Great Value Whole Milk?',
    'context':  'Great Value Whole Milk 1 gallon is priced at $3.98 and is located in Aisle 12.',
    'expected': 'Great Value Whole Milk 1 gallon costs $3.98 and is in Aisle 12.',
}
answers = {
    'correct':      'Great Value Whole Milk costs $3.98 and can be found in Aisle 12.',
    'hallucinated': 'Great Value Whole Milk costs $2.49 and is on sale this week.',
    'incomplete':   'Great Value Whole Milk is available at Walmart.',
}
print('LLM-as-Judge Demo:')
print()
for label, answer in answers.items():
    r = llm_judge(demo['query'], demo['context'], demo['expected'], answer)
    print(f'  [{label:<13}] Score: {r["score"]}/3 | Normalised: {r["normalised"]} | {r["label"]}')
    print(f'                 Reason: {r["reason"]}')
    print()

LLM-as-Judge Demo:

  [correct      ] Score: 2/3 | Normalised: 0.667 | Correct but incomplete
                 Reason: The response correctly identifies the price and location of the Great Value Whole Milk but omits the detail that it is for a 1 gallon size, which is a necessary detail for complete accuracy.

  [hallucinated ] Score: 0/3 | Normalised: 0.0 | Wrong or harmful
                 Reason: The response is factually incorrect as it provides a wrong price and sale status not mentioned in the ground truth.

  [incomplete   ] Score: 1/3 | Normalised: 0.333 | Partially correct
                 Reason: The response indicates the correct store but fails to provide the specific price and location within the store, which are key details needed by the customer.



## Section 2: Benchmark Design

### What is a Benchmark?

A benchmark is a standardised, reproducible test suite used to compare AI systems,
model versions, or configuration changes against a consistent baseline.

**A benchmark is different from a golden dataset:**

| Concept | Golden Dataset | Benchmark |
|---|---|---|
| Purpose | Measure current quality | Compare systems over time |
| Contents | Query + context + expected answer | Golden dataset + metrics + thresholds + baseline scores |
| Output | Pass/fail per metric | Delta report: improved / regressed / stable |
| Versioned? | Yes, but slowly | Yes, with strict change control |

**Benchmark design principles for Walmart Retail Assistant:**

1. **Category balance:** Equal representation across price, policy, order, hours, multi-step
2. **Difficulty tiers:** Easy (direct lookup), medium (comparison), hard (multi-intent, edge case)
3. **Stable test set:** Never change golden records mid-evaluation; version them instead
4. **Minimum size:** 20 records for dev; 200+ for production regression gates
5. **Threshold documentation:** Every metric has an explicit pass/fail threshold

**Regression test trigger conditions:**
- Any model version change (e.g. gpt-4-turbo -> gpt-4o-mini)
- Any prompt version change
- Any retrieval configuration change (chunk size, K, embedding model)
- Weekly scheduled run to detect silent drift

In [6]:
# Compact benchmark dataset: 10 records across 5 categories
# (subset of IN10 golden dataset -- same format, fewer records for API cost control)

BENCHMARK = [
    {'id':'B001','cat':'price',
     'query':'What is the price of Great Value Whole Milk?',
     'context':'Great Value Whole Milk 1 gallon is priced at $3.98, located in Aisle 12.',
     'expected':'Great Value Whole Milk 1 gallon costs $3.98 and is in Aisle 12.'},
    {'id':'B002','cat':'price',
     'query':'How much does Great Value Eggs 12-count cost?',
     'context':'Great Value Large Eggs 12-count is $2.78, dairy section, Aisle 8.',
     'expected':'Great Value Large Eggs 12-count is $2.78, in the dairy section (Aisle 8).'},
    {'id':'B003','cat':'policy',
     'query':'What is the return policy for electronics?',
     'context':'Electronics must be returned within 30 days with receipt and original packaging.',
     'expected':'Electronics must be returned within 30 days with receipt and in original packaging.'},
    {'id':'B004','cat':'policy',
     'query':'Can I return an opened item without a receipt?',
     'context':'Returns without receipt accepted within 90 days. Valid photo ID required. Refund as store credit.',
     'expected':'Yes, returns without receipt are accepted within 90 days with a valid photo ID. Refund is issued as store credit.'},
    {'id':'B005','cat':'order',
     'query':'Where is my order ORD-78901?',
     'context':'Order ORD-78901: shipped via FedEx, tracking FX123456, estimated delivery July 3 2026.',
     'expected':'Order ORD-78901 has been shipped via FedEx (FX123456), arriving by July 3, 2026.'},
    {'id':'B006','cat':'order',
     'query':'How do I track my Walmart+ delivery?',
     'context':'Track Walmart+ deliveries in the Walmart app under Orders or at walmart.com/account/orders.',
     'expected':'Track your Walmart+ delivery in the Walmart app under Orders or at walmart.com/account/orders.'},
    {'id':'B007','cat':'hours',
     'query':'What time does Walmart open?',
     'context':'Most Walmart Supercenters open at 6:00 AM and close at 11:00 PM daily.',
     'expected':'Most Walmart Supercenters open at 6:00 AM and close at 11:00 PM daily.'},
    {'id':'B008','cat':'hours',
     'query':'Is Walmart open on Thanksgiving?',
     'context':'Walmart stores are closed on Thanksgiving Day. They reopen at 6:00 AM on Black Friday.',
     'expected':'Walmart stores are closed on Thanksgiving Day and reopen at 6:00 AM on Black Friday.'},
    {'id':'B009','cat':'multi_step',
     'query':'Is Great Value Milk in stock and what aisle is it in?',
     'context':'Great Value Whole Milk 1 gallon: in stock (47 units), Aisle 12, Dairy section.',
     'expected':'Great Value Whole Milk is in stock with 47 units in Aisle 12 (Dairy section).'},
    {'id':'B010','cat':'multi_step',
     'query':'Compare Great Value and Tide laundry detergent on price and size.',
     'context':'Great Value 150 oz is $8.97 (6c/oz). Tide Original 92 oz is $11.97 (13c/oz).',
     'expected':'Great Value (150 oz) costs $8.97 at 6c/oz. Tide (92 oz) costs $11.97 at 13c/oz. Great Value offers more volume at half the per-ounce cost.'},
]
print(f'Benchmark loaded: {len(BENCHMARK)} records across {len(set(r["cat"] for r in BENCHMARK))} categories')

Benchmark loaded: 10 records across 5 categories


## Section 3: Baseline Evaluation Run

### What is a Baseline?

A baseline is the first complete evaluation run of a system. Every future evaluation
is compared against it. The baseline establishes what 'normal' looks like.

**Baseline selection criteria:**
- Run on the full benchmark dataset, not a sample
- Use the current production model and prompt version
- Record all metric scores alongside the model version and timestamp
- Store in version control alongside the code

**For this lab:** Baseline = gpt-4-turbo answers scored by LLM-as-judge.
The Walmart assistant answers each query, the judge scores each answer.

In [7]:
BASELINE_MODEL = 'gpt-4-turbo'
SYSTEM_PROMPT  = (
    'You are the Walmart Retail Assistant. Help customers with product information, '
    'store policies, and order enquiries. Be concise and accurate. '
    'Base your answer only on the provided context.'
)

def run_assistant(query: str, context: str, model: str) -> dict:
    """Run the Walmart assistant and return answer with latency."""
    start = time.time()
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': f'Context: {context}\n\nQuestion: {query}'},
        ],
        temperature=0,
        max_tokens=200,
        timeout=10,
    )
    return {
        'answer':      resp.choices[0].message.content,
        'latency_ms':  round((time.time() - start) * 1000),
        'input_tokens':  resp.usage.prompt_tokens,
        'output_tokens': resp.usage.completion_tokens,
    }

def run_benchmark(benchmark: list, model: str, label: str) -> list:
    results = []
    print(f'Running benchmark: {label} ({model})')
    for rec in benchmark:
        assistant = run_assistant(rec['query'], rec['context'], model)
        judge     = llm_judge(rec['query'], rec['context'], rec['expected'], assistant['answer'])
        results.append({
            'id':           rec['id'],
            'category':     rec['cat'],
            'model':        model,
            'answer':       assistant['answer'],
            'latency_ms':   assistant['latency_ms'],
            'input_tokens': assistant['input_tokens'],
            'output_tokens':assistant['output_tokens'],
            'judge_score':  judge['score'],
            'normalised':   judge['normalised'],
            'label':        judge['label'],
            'reason':       judge['reason'],
        })
        print(f'  [{rec["id"]}] score={judge["score"]}/3 | {judge["label"]} | {assistant["latency_ms"]}ms')
    return results

baseline_results = run_benchmark(BENCHMARK, BASELINE_MODEL, 'Baseline')

baseline_avg = round(sum(r['normalised'] for r in baseline_results) / len(baseline_results), 3)
baseline_lat = round(sum(r['latency_ms'] for r in baseline_results) / len(baseline_results))
baseline_tok = round(sum(r['input_tokens'] + r['output_tokens'] for r in baseline_results) / len(baseline_results))
print(f'\nBaseline avg normalised score: {baseline_avg}')
print(f'Baseline avg latency         : {baseline_lat} ms')
print(f'Baseline avg tokens/call     : {baseline_tok}')

Running benchmark: Baseline (gpt-4-turbo)
  [B001] score=2/3 | Correct but incomplete | 1649ms
  [B002] score=2/3 | Correct but incomplete | 1844ms
  [B003] score=3/3 | Complete and correct | 1057ms
  [B004] score=3/3 | Complete and correct | 2240ms
  [B005] score=2/3 | Correct but incomplete | 2844ms
  [B006] score=3/3 | Complete and correct | 1985ms
  [B007] score=2/3 | Correct but incomplete | 1429ms
  [B008] score=3/3 | Complete and correct | 1877ms
  [B009] score=3/3 | Complete and correct | 1878ms
  [B010] score=3/3 | Complete and correct | 3256ms

Baseline avg normalised score: 0.867
Baseline avg latency         : 2006 ms
Baseline avg tokens/call     : 112


## Section 4: Regression Testing

### What is Regression Testing?

Regression testing is the practice of re-running the benchmark after any change
to the system and comparing the new scores against the baseline to detect quality degradation.

**Regression detection rule:**
```
Score delta = comparison_score - baseline_score

If score_delta < -0.05 for any individual metric: REGRESSION DETECTED
If aggregate_delta < -0.03: REGRESSION DETECTED
If aggregate_delta >= 0.00: STABLE or IMPROVED
```

**Why the threshold is -0.05 (not 0):**
LLM judge scores have natural variance. A drop of 0.02 may be noise.
A drop of 0.05 or more on a normalised scale indicates a real degradation.

**Go/No-Go decision matrix:**

| Aggregate Delta | Category Delta | Decision |
|---|---|---|
| >= 0.00 | Any | APPROVED -- deploy |
| -0.03 to -0.01 | No category < -0.05 | CONDITIONAL -- review in staging |
| < -0.03 | Any | BLOCKED -- do not deploy |
| Any | Any category < -0.10 | BLOCKED -- critical regression |

**What to remember:**
- Always compare against the same baseline. If you update the baseline, record why.
- Run regression tests in CI/CD. Every PR that touches the prompt or model config
  should trigger an automated benchmark run.
- Track cost and latency deltas alongside quality deltas. A quality improvement that
  triples cost is not necessarily a win.

In [8]:
COMPARISON_MODEL = 'gpt-4o-mini'
comparison_results = run_benchmark(BENCHMARK, COMPARISON_MODEL, 'Comparison')

comp_avg = round(sum(r['normalised'] for r in comparison_results) / len(comparison_results), 3)
comp_lat = round(sum(r['latency_ms'] for r in comparison_results) / len(comparison_results))
comp_tok = round(sum(r['input_tokens'] + r['output_tokens'] for r in comparison_results) / len(comparison_results))

print(f'\nComparison avg normalised score: {comp_avg}')
print(f'Comparison avg latency         : {comp_lat} ms')
print(f'Comparison avg tokens/call     : {comp_tok}')

Running benchmark: Comparison (gpt-4o-mini)
  [B001] score=2/3 | Correct but incomplete | 1807ms
  [B002] score=2/3 | Correct but incomplete | 770ms
  [B003] score=3/3 | Complete and correct | 806ms
  [B004] score=3/3 | Complete and correct | 947ms
  [B005] score=3/3 | Complete and correct | 819ms
  [B006] score=3/3 | Complete and correct | 927ms
  [B007] score=2/3 | Correct but incomplete | 807ms
  [B008] score=2/3 | Correct but incomplete | 694ms
  [B009] score=3/3 | Complete and correct | 1142ms
  [B010] score=3/3 | Complete and correct | 1698ms

Comparison avg normalised score: 0.867
Comparison avg latency         : 1042 ms
Comparison avg tokens/call     : 106


## Section 5: Score Delta Analysis

### What is Score Delta Analysis?

Score delta analysis compares per-record and per-category scores between the baseline
and comparison run to identify exactly where quality improved or degraded.

**Three levels of delta analysis:**

| Level | What it measures | When to use |
|---|---|---|
| Aggregate delta | Overall quality shift | Go/no-go gate |
| Category delta | Which query types regressed | Targeted investigation |
| Per-record delta | Exact queries that changed | Root-cause debugging |

**Drift threshold:** Flag any per-record delta < -0.33 (i.e., dropped at least 1 full
point on the 0-3 scale). These are the highest-priority cases to debug.

In [9]:
# Build comparison table
baseline_map    = {r['id']: r for r in baseline_results}
comparison_map  = {r['id']: r for r in comparison_results}

deltas = []
for rid in baseline_map:
    b = baseline_map[rid]
    c = comparison_map[rid]
    delta = round(c['normalised'] - b['normalised'], 3)
    deltas.append({
        'id':       rid,
        'category': b['category'],
        'baseline': b['normalised'],
        'comparison': c['normalised'],
        'delta':    delta,
        'latency_delta_ms': c['latency_ms'] - b['latency_ms'],
        'status':   'REGRESSED' if delta < -0.05 else ('IMPROVED' if delta > 0.05 else 'STABLE'),
    })

aggregate_delta = round(comp_avg - baseline_avg, 3)

print('Score Delta Analysis:')
print('=' * 65)
print(f'Baseline  ({BASELINE_MODEL}): {baseline_avg}')
print(f'Comparison ({COMPARISON_MODEL}): {comp_avg}')
print(f'Aggregate delta            : {aggregate_delta:+.3f}')
print()
print(f'{"ID":<6} {"Category":<12} {"Baseline":<10} {"Comparison":<12} {"Delta":<8} Status')
print('-' * 65)
for d in deltas:
    print(f'{d["id"]:<6} {d["category"]:<12} {d["baseline"]:<10} {d["comparison"]:<12} {d["delta"]:+.3f}   {d["status"]}')

# Category-level summary
from collections import defaultdict
cat_deltas = defaultdict(list)
for d in deltas:
    cat_deltas[d['category']].append(d['delta'])
print()
print('Category delta summary:')
for cat, ds in sorted(cat_deltas.items()):
    avg_d = round(sum(ds)/len(ds), 3)
    flag  = ' <-- INVESTIGATE' if avg_d < -0.05 else ''
    print(f'  {cat:<12}: {avg_d:+.3f}{flag}')

Score Delta Analysis:
Baseline  (gpt-4-turbo): 0.867
Comparison (gpt-4o-mini): 0.867
Aggregate delta            : +0.000

ID     Category     Baseline   Comparison   Delta    Status
-----------------------------------------------------------------
B001   price        0.667      0.667        +0.000   STABLE
B002   price        0.667      0.667        +0.000   STABLE
B003   policy       1.0        1.0          +0.000   STABLE
B004   policy       1.0        1.0          +0.000   STABLE
B005   order        0.667      1.0          +0.333   IMPROVED
B006   order        1.0        1.0          +0.000   STABLE
B007   hours        0.667      0.667        +0.000   STABLE
B008   hours        1.0        0.667        -0.333   REGRESSED
B009   multi_step   1.0        1.0          +0.000   STABLE
B010   multi_step   1.0        1.0          +0.000   STABLE

Category delta summary:
  hours       : -0.167 <-- INVESTIGATE
  multi_step  : +0.000
  order       : +0.167
  policy      : +0.000
  price       

In [10]:
# GO / NO-GO DECISION ENGINE

REGRESSION_THRESHOLD     = -0.05   # per-record delta
AGGREGATE_THRESHOLD      = -0.03   # overall aggregate delta
CRITICAL_THRESHOLD       = -0.10   # any category

regressions = [d for d in deltas if d['delta'] < REGRESSION_THRESHOLD]
improvements = [d for d in deltas if d['delta'] > 0.05]

cat_avg_deltas = {}
for cat, ds in cat_deltas.items():
    cat_avg_deltas[cat] = round(sum(ds)/len(ds), 3)

critical_cat = [c for c, d in cat_avg_deltas.items() if d < CRITICAL_THRESHOLD]

if critical_cat:
    decision = 'BLOCKED -- critical category regression'
elif aggregate_delta < -0.03:
    decision = 'BLOCKED -- aggregate quality regression'
elif aggregate_delta < 0 and regressions:
    decision = 'CONDITIONAL -- minor regression, review in staging before deploy'
else:
    decision = 'APPROVED -- deploy to production'

print('GO / NO-GO DECISION REPORT')
print('=' * 55)
print(f'Baseline model    : {BASELINE_MODEL}')
print(f'Comparison model  : {COMPARISON_MODEL}')
print(f'Aggregate delta   : {aggregate_delta:+.3f} (threshold: {AGGREGATE_THRESHOLD})')
print(f'Regressions       : {len(regressions)} record(s)')
print(f'Improvements      : {len(improvements)} record(s)')
print(f'Critical categories: {critical_cat if critical_cat else "None"}')
print()
print(f'Latency delta     : {comp_lat - baseline_lat:+d} ms (avg per call)')
print(f'Token delta       : {comp_tok - baseline_tok:+d} tokens (avg per call)')
print()
print(f'DECISION: {decision}')

# Save benchmark report
report = {
    'baseline_model':    BASELINE_MODEL,
    'comparison_model':  COMPARISON_MODEL,
    'baseline_avg_score': baseline_avg,
    'comparison_avg_score': comp_avg,
    'aggregate_delta':   aggregate_delta,
    'baseline_avg_latency_ms':    baseline_lat,
    'comparison_avg_latency_ms':  comp_lat,
    'latency_delta_ms':  comp_lat - baseline_lat,
    'regressions':       len(regressions),
    'improvements':      len(improvements),
    'critical_categories': critical_cat,
    'category_deltas':   cat_avg_deltas,
    'decision':          decision,
    'per_record_deltas': deltas,
}
Path('in11_benchmark_report.json').write_text(json.dumps(report, indent=2))
print()
print('Report saved to in11_benchmark_report.json')

GO / NO-GO DECISION REPORT
Baseline model    : gpt-4-turbo
Comparison model  : gpt-4o-mini
Aggregate delta   : +0.000 (threshold: -0.03)
Regressions       : 1 record(s)
Improvements      : 1 record(s)
Critical categories: ['hours']

Latency delta     : -964 ms (avg per call)
Token delta       : -6 tokens (avg per call)

DECISION: BLOCKED -- critical category regression

Report saved to in11_benchmark_report.json


## Section 6: Comparative LLM-as-Judge (Side-by-Side Scoring)

### What is Comparative Evaluation?

In comparative evaluation, the judge sees both the baseline and comparison answers
simultaneously and picks which is better -- or declares a tie.
This is more sensitive than scoring each answer independently because it eliminates
the judge's internal calibration variance.

**Positional bias mitigation:** Run each pair twice with the answer order swapped.
If the judge picks Answer A in run 1 and Answer B in run 2, record it as a tie.

**Win rate interpretation:**

| Comparison Win Rate | Interpretation |
|---|---|
| > 60% | Comparison model clearly better |
| 45 - 60% | Roughly equivalent -- choose on cost/latency |
| < 45% | Baseline is better -- do not deploy comparison |

In [11]:
def comparative_judge(query: str, context: str, answer_a: str, answer_b: str,
                       label_a: str = 'A', label_b: str = 'B') -> dict:
    """Compare two answers. Returns winner (A, B, or TIE) and reason."""
    prompt = (
        f'Query: {query}\n'
        f'Context: {context}\n\n'
        f'Answer {label_a}: {answer_a}\n\n'
        f'Answer {label_b}: {answer_b}\n\n'
        f'Which answer is better for a Walmart retail customer? '
        f'Consider: accuracy, completeness, and conciseness. '
        f'Return JSON: {{"winner": "{label_a}" or "{label_b}" or "TIE", "reason": "one sentence"}}'
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a retail AI evaluator. Be objective.'},
            {'role': 'user',   'content': prompt},
        ],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    result = json.loads(resp.choices[0].message.content)
    return {'winner': result.get('winner', 'TIE'), 'reason': result.get('reason', '')}

# Run comparative evaluation on first 4 records
print('Comparative Evaluation (4 records, positional bias mitigated):')
print()
wins = {BASELINE_MODEL: 0, COMPARISON_MODEL: 0, 'TIE': 0}
for rec in BENCHMARK[:4]:
    ans_b = baseline_map[rec['id']]['answer']
    ans_c = comparison_map[rec['id']]['answer']

    # Order 1: baseline=A, comparison=B
    r1 = comparative_judge(rec['query'], rec['context'], ans_b, ans_c, 'BASELINE', 'COMPARISON')
    # Order 2: comparison=A, baseline=B (swap to detect positional bias)
    r2 = comparative_judge(rec['query'], rec['context'], ans_c, ans_b, 'COMPARISON', 'BASELINE')

    # If both runs agree: accept that winner. If they disagree: TIE
    if r1['winner'] == r2['winner']:
        winner = BASELINE_MODEL if r1['winner'] == 'BASELINE' else COMPARISON_MODEL
    elif r1['winner'] == 'TIE' or r2['winner'] == 'TIE':
        winner = 'TIE'
    else:
        winner = 'TIE'  # disagreement after swap = positional bias = TIE

    wins[winner] = wins.get(winner, 0) + 1
    print(f'  [{rec["id"]}] Winner: {winner}')
    print(f'         Run1: {r1["winner"]} | Run2: {r2["winner"]}')
    print(f'         {r1["reason"]}')
    print()

total = sum(wins.values())
print(f'Comparative Results ({total} records):')
for model, count in wins.items():
    pct = round(count/total*100)
    print(f'  {model}: {count} wins ({pct}%)')

Comparative Evaluation (4 records, positional bias mitigated):

  [B001] Winner: TIE
         Run1: COMPARISON | Run2: TIE
         The COMPARISON answer is slightly clearer due to the inclusion of parentheses around the quantity, enhancing readability.

  [B002] Winner: gpt-4-turbo
         Run1: BASELINE | Run2: BASELINE
         The BASELINE answer is more grammatically correct and clear, using 'costs' instead of 'cost', which is more appropriate for a singular subject.

  [B003] Winner: TIE
         Run1: COMPARISON | Run2: TIE
         It is more grammatically correct and concise, enhancing clarity for the customer.

  [B004] Winner: gpt-4o-mini
         Run1: TIE | Run2: TIE
         Both answers accurately and concisely convey the same essential information regarding the return policy.

Comparative Results (4 records):
  gpt-4-turbo: 1 wins (25%)
  gpt-4o-mini: 1 wins (25%)
  TIE: 2 wins (50%)


## Summary

| Concept | Key Rule |
|---|---|
| LLM-as-judge rubric | 0-3 scale; normalised to 0.0-1.0; judge temperature must be 0 |
| Baseline | First complete run; all future runs compared against it |
| Regression threshold | Per-record delta < -0.05; aggregate delta < -0.03 |
| Critical threshold | Any category delta < -0.10 = hard block |
| Go/No-Go | APPROVED / CONDITIONAL / BLOCKED based on aggregate + category deltas |
| Comparative judge | Run twice with swapped order; disagreement = TIE (positional bias) |
| Track cost and latency | Quality win that triples cost is not always a net win |

**Next: IN12** -- Logging architecture, tracing, Langfuse, debugging agentic workflows,
and generating the final `evaluation_scorecard.txt` + `observability_dashboard_design.txt`.